# result

统计 `base_latest` 各 domain 在 `base_latest_output` 下的 `*_evaluate.json` 指标：

- `cell_recall`, `cell_precision`, `cell_f1`
- `entity_*`, `relation_*`, `llm_score`

In [ ]:
import json
from pathlib import Path

BASE_LATEST = Path('dataset/case/base_latest')
BASE_LATEST_OUTPUT = Path('dataset/case/base_latest_output/llm')

# 以 base_latest 的一级目录作为 domain 列表
DOMAINS = sorted([p.name for p in BASE_LATEST.iterdir() if p.is_dir()])
print('domains:', DOMAINS)

def avg(values):
    nums = [v for v in values if isinstance(v, (int, float))]
    return sum(nums) / len(nums) if nums else None


def parse_model_name(stem: str) -> str:
    """从文件名去掉 evaluate 相关后缀，得到模型标签。"""
    for suf in ('_evaluate_pipeline_oracle', '_evaluate'):
        if stem.endswith(suf):
            return stem[:-len(suf)]
    return stem


records = []
for domain in DOMAINS:
    domain_dir = BASE_LATEST_OUTPUT / domain
    if not domain_dir.exists():
        continue

    for case_dir in sorted([p for p in domain_dir.iterdir() if p.is_dir()]):
        # evaluate_files = sorted(case_dir.glob('*/*_evaluate.json'))
        evaluate_files = sorted(case_dir.glob('*/*_evaluate_pipeline_oracle.json'))

        for eval_file in evaluate_files:
            with eval_file.open('r', encoding='utf-8') as f:
                data = json.load(f)

            records.append({
                'domain': domain,
                'case': case_dir.name,
                'model': parse_model_name(eval_file.stem),
                'entity_precision': data.get('entity_precision'),
                'entity_recall': data.get('entity_recall'),
                'entity_f1': data.get('entity_f1'),
                'relation_precision': data.get('relation_precision'),
                'relation_recall': data.get('relation_recall'),
                'relation_f1': data.get('relation_f1'),
                'cell_precision': data.get('cell_precision'),
                'cell_recall': data.get('cell_recall'),
                'cell_f1': data.get('cell_f1'),
                'llm_score': data.get('llm_score'),
                'file': str(eval_file),
            })

print('loaded records:', len(records))

In [ ]:
# ====== relation-level 统计（整目录汇总）======
import json
from pathlib import Path
from collections import Counter
import pandas as pd
# 用户指定目录（若不存在则回退到常见目录）
CASE_ROOT = Path("dataset/case/base_latest")

meta_files = sorted([p for p in CASE_ROOT.rglob("meta.json") if p.is_file()])
print(f"扫描目录: {CASE_ROOT}")
print(f"发现 meta.json 数量: {len(meta_files)}")

all_type_counter = Counter()
relation_rows = []

for meta_path in meta_files:
    with open(meta_path, "r", encoding="utf-8") as fp:
        meta = json.load(fp)

    type_map = meta.get("type", {}) or {}
    if not isinstance(type_map, dict):
        continue

    all_type_counter.update(str(v).strip() for v in type_map.values())

    parts = meta_path.parts
    # 兼容 .../<domain>/<case>/meta.json 或 .../llm/<domain>/<case>/.../meta.json
    domain = parts[-3] if len(parts) >= 3 else "unknown"
    case_name = parts[-2] if len(parts) >= 2 else "unknown"

    for table_name, rel_type in type_map.items():
        relation_rows.append({
            "domain": domain,
            "case": case_name,
            "table": str(table_name),
            "relation_type": str(rel_type).strip(),
            "meta_path": str(meta_path),
        })

print("\n=== 原始关系类型总计 ===")
print(all_type_counter)

relation_type_df = pd.DataFrame(relation_rows)
if relation_type_df.empty:
    print("未找到可用的关系类型数据")
else:
    def normalize_relation_type_list(raw_value):
        """
        支持 relation_type 为：
        - 字符串: "1:2" / "2-hop"
        - 列表: ["1:2", "2-hop"]
        - 其他可序列化类型
        """
        if isinstance(raw_value, list):
            vals = raw_value
        else:
            text = str(raw_value).strip()
            if text.startswith("[") and text.endswith("]"):
                try:
                    parsed = json.loads(text.replace("'", '"'))
                    vals = parsed if isinstance(parsed, list) else [text]
                except Exception:
                    vals = [text]
            else:
                vals = [text]

        out = []
        for v in vals:
            s = str(v).strip().lower()
            if s:
                out.append(s)
        return out

    def map_relation_bucket(type_token: str):
        s = str(type_token).strip().lower()
        if s in {"1:1", "one-to-one", "o2o"}:
            return "1:1"
        if s in {"2-hop", "multi-hop", "multihop", "hop2", "2hop"}:
            return "multi-hop"
        # 将 1:2 / 1:n / n:1 / m:n 等统一归入 1:N
        if ":" in s and s != "1:1":
            return "1:N"
        if s in {"1:n", "n:1", "m:n", "many-to-many", "one-to-many", "many-to-one"}:
            return "1:N"
        return "other"

    relation_type_df = relation_type_df.copy()
    relation_type_df["relation_type_tokens"] = relation_type_df["relation_type"].map(normalize_relation_type_list)
    relation_type_df["relation_buckets"] = relation_type_df["relation_type_tokens"].map(
        lambda xs: sorted({map_relation_bucket(x) for x in xs})
    )

    exploded_bucket_df = relation_type_df.explode("relation_buckets", ignore_index=True)

    print("\n=== 各原始关系类型数量 ===")
    display(
        relation_type_df.groupby("relation_type", as_index=False)
        .size()
        .rename(columns={"size": "count"})
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )

    bucket_count_df = (
        exploded_bucket_df.groupby("relation_buckets", as_index=False)
        .size()
        .rename(columns={"relation_buckets": "bucket", "size": "count"})
    )

    # 强制展示目标三类，即使计数为 0 也显示
    target_buckets = ["1:1", "1:N", "multi-hop", "other"]
    bucket_count_df = (
        pd.DataFrame({"bucket": target_buckets})
        .merge(bucket_count_df, on="bucket", how="left")
        .fillna({"count": 0})
    )
    bucket_count_df["count"] = bucket_count_df["count"].astype(int)

    print("\n=== 分类后数量（1:1 / 1:N / multi-hop，可重复计数）===")
    display(bucket_count_df)
    print("分类计数字典:", dict(zip(bucket_count_df["bucket"], bucket_count_df["count"])))

    print("\n=== 关系表明细（含分类，前50）===")
    display(
        relation_type_df[["domain", "case", "table", "relation_type", "relation_buckets", "meta_path"]]
        .sort_values(["domain", "case", "table"])
        .head(5)
        .reset_index(drop=True)
    )

In [ ]:
import pandas as pd

# 按模型过滤，例如 gpt-4o / gpt-5.4（两边统一把 - 规范成 _）
TARGET_MODEL = 'qwen2.5-72b-instruct'
model_key = TARGET_MODEL.lower().replace('-', '_')

model_records = [
    r for r in records
    if model_key in (r.get('model') or '').lower().replace('-', '_')
]

print(f'=== model={TARGET_MODEL} 匹配文件数 ===')
print(len(model_records))

if not model_records:
    print('未找到对应模型的 evaluate.json')
else:
    model_cols = [
        'entity_precision', 'entity_recall', 'entity_f1',
        'relation_precision', 'relation_recall', 'relation_f1',
        'cell_precision', 'cell_recall', 'cell_f1',
        'llm_score',
    ]

    # 1) 整体均值表
    overall_metrics = {col: avg([r.get(col, 0) for r in model_records]) for col in model_cols}
    overall_metrics['end2end_precision'] = overall_metrics.pop('cell_precision')
    overall_metrics['end2end_recall'] = overall_metrics.pop('cell_recall')
    overall_metrics['end2end_f1'] = overall_metrics.pop('cell_f1')

    overall_df = pd.DataFrame([overall_metrics], index=['overall'])
    display(overall_df.round(4))

    # 2) 按 domain 均值表
    domain_rows_out = []
    for domain in sorted({r.get('domain', 'unknown') for r in model_records}):
        domain_rows = [r for r in model_records if r.get('domain') == domain]
        metrics = {col: avg([r.get(col, 0) for r in domain_rows]) for col in model_cols}
        metrics['end2end_precision'] = metrics.pop('cell_precision')
        metrics['end2end_recall'] = metrics.pop('cell_recall')
        metrics['end2end_f1'] = metrics.pop('cell_f1')
        metrics['domain'] = domain
        domain_rows_out.append(metrics)

    domain_df = pd.DataFrame(domain_rows_out)
    domain_df = domain_df[
        ['domain',
         'entity_precision', 'entity_recall', 'entity_f1',
         'relation_precision', 'relation_recall', 'relation_f1',
         'end2end_precision', 'end2end_recall', 'end2end_f1',
         'llm_score']
    ]
    display(domain_df.round(4))

    # 3) 在三种关系表分类（1:1 / 1:N / multi-hop）上的结果
    target_buckets = ["1:1", "1:N", "multi-hop"]

    # 优先复用前序单元产出的 relation_type_df；若不存在则临时构建
    if "relation_type_df" not in globals() or relation_type_df is None or relation_type_df.empty:
        tmp_rows = []
        for domain in DOMAINS:
            for meta_path in sorted((BASE_LATEST / domain).glob("case*/meta.json")):
                case_name = meta_path.parent.name
                with meta_path.open("r", encoding="utf-8") as fp:
                    meta = json.load(fp)
                type_map = meta.get("type", {}) or {}
                if not isinstance(type_map, dict):
                    continue
                for table_name, rel_type in type_map.items():
                    tmp_rows.append(
                        {
                            "domain": domain,
                            "case": case_name,
                            "table": str(table_name),
                            "relation_type": str(rel_type).strip(),
                        }
                    )
        relation_type_df_local = pd.DataFrame(tmp_rows)
    else:
        relation_type_df_local = relation_type_df.copy()

    def _normalize_relation_type_list(raw_value):
        if isinstance(raw_value, list):
            vals = raw_value
        else:
            text = str(raw_value).strip()
            if text.startswith("[") and text.endswith("]"):
                try:
                    parsed = json.loads(text.replace("'", '"'))
                    vals = parsed if isinstance(parsed, list) else [text]
                except Exception:
                    vals = [text]
            else:
                vals = [text]
        return [str(v).strip().lower() for v in vals if str(v).strip()]

    def _map_relation_bucket(type_token: str):
        s = str(type_token).strip().lower()
        if s in {"1:1", "one-to-one", "o2o", "1;1"}:
            return "1:1"
        if s in {"2-hop", "multi-hop", "multihop", "hop2", "2hop"}:
            return "multi-hop"
        if ":" in s and s != "1:1":
            return "1:N"
        if s in {"1:n", "n:1", "m:n", "many-to-many", "one-to-many", "many-to-one", "many2one"}:
            return "1:N"
        return "other"

    bucket_metrics_rows = []
    case_bucket_df = pd.DataFrame(columns=["domain", "case", "relation_buckets"])
    if not relation_type_df_local.empty:
        relation_type_df_local = relation_type_df_local.copy()
        relation_type_df_local["relation_type_tokens"] = relation_type_df_local["relation_type"].map(_normalize_relation_type_list)
        relation_type_df_local["relation_buckets"] = relation_type_df_local["relation_type_tokens"].map(
            lambda xs: sorted({_map_relation_bucket(x) for x in xs})
        )
        case_bucket_df = relation_type_df_local[["domain", "case", "relation_buckets"]].explode("relation_buckets", ignore_index=True)

        for bucket in target_buckets:
            bucket_cases_df = case_bucket_df[case_bucket_df["relation_buckets"] == bucket][["domain", "case"]].drop_duplicates()
            bucket_case_pairs = {(r.domain, r.case) for r in bucket_cases_df.itertuples(index=False)}
            bucket_records = [
                r for r in model_records
                if (r.get("domain"), r.get("case")) in bucket_case_pairs
            ]

            if bucket_records:
                metrics = {col: avg([r.get(col, 0) for r in bucket_records]) for col in model_cols}
                metrics["end2end_precision"] = metrics.pop("cell_precision")
                metrics["end2end_recall"] = metrics.pop("cell_recall")
                metrics["end2end_f1"] = metrics.pop("cell_f1")
            else:
                metrics = {
                    "entity_precision": None,
                    "entity_recall": None,
                    "entity_f1": None,
                    "relation_precision": None,
                    "relation_recall": None,
                    "relation_f1": None,
                    "llm_score": None,
                    "end2end_precision": None,
                    "end2end_recall": None,
                    "end2end_f1": None,
                }

            metrics["relation_bucket"] = bucket
            metrics["bucket_case_count"] = len(bucket_case_pairs)
            metrics["matched_file_count"] = len(bucket_records)
            bucket_metrics_rows.append(metrics)

    if bucket_metrics_rows:
        bucket_metrics_df = pd.DataFrame(bucket_metrics_rows)[[
            "relation_bucket", "bucket_case_count", "matched_file_count",
            "entity_precision", "entity_recall", "entity_f1",
            "relation_precision", "relation_recall", "relation_f1",
            "end2end_precision", "end2end_recall", "end2end_f1",
            "llm_score",
        ]]
        print("\n=== 三种关系表分类上的结果 ===")
        display(bucket_metrics_df.round(4))

        # 3.1) 按 bucket × domain 的指标
        bucket_domain_rows = []
        model_df = pd.DataFrame(model_records)
        for bucket in target_buckets:
            bucket_cases = case_bucket_df[case_bucket_df["relation_buckets"] == bucket][["domain", "case"]].drop_duplicates()
            for domain in sorted(bucket_cases["domain"].dropna().unique().tolist()):
                domain_cases = set(
                    tuple(x)
                    for x in bucket_cases[bucket_cases["domain"] == domain][["domain", "case"]].to_records(index=False)
                )
                sub_df = model_df[
                    model_df.apply(lambda r: (r.get("domain"), r.get("case")) in domain_cases, axis=1)
                ]
                if sub_df.empty:
                    continue
                row = {
                    "relation_bucket": bucket,
                    "domain": domain,
                    "matched_file_count": int(len(sub_df)),
                }
                row["entity_f1"] = avg(sub_df["entity_f1"].tolist())
                row["relation_f1"] = avg(sub_df["relation_f1"].tolist())
                row["end2end_f1"] = avg(sub_df["cell_f1"].tolist())
                row["llm_score"] = avg(sub_df["llm_score"].tolist())
                bucket_domain_rows.append(row)

        if bucket_domain_rows:
            bucket_domain_df = pd.DataFrame(bucket_domain_rows).sort_values(
                ["relation_bucket", "end2end_f1"], ascending=[True, False]
            ).reset_index(drop=True)
            print("\n=== bucket × domain 指标（按 end2end_f1）===")
            display(bucket_domain_df.round(4))

        # 3.2) 每个 bucket 最差 case 明细（按 end2end_f1 升序）
        bucket_worst_rows = []
        for bucket in target_buckets:
            bucket_cases = case_bucket_df[case_bucket_df["relation_buckets"] == bucket][["domain", "case"]].drop_duplicates()
            case_pairs = set(tuple(x) for x in bucket_cases[["domain", "case"]].to_records(index=False))
            sub_df = model_df[
                model_df.apply(lambda r: (r.get("domain"), r.get("case")) in case_pairs, axis=1)
            ]
            if sub_df.empty:
                continue
            case_metric_df = (
                sub_df.groupby(["domain", "case"], as_index=False)
                .agg(
                    entity_f1=("entity_f1", "mean"),
                    relation_f1=("relation_f1", "mean"),
                    end2end_f1=("cell_f1", "mean"),
                    llm_score=("llm_score", "mean"),
                    matched_file_count=("file", "count"),
                )
                .sort_values("end2end_f1", ascending=True)
            )
            case_metric_df["relation_bucket"] = bucket
            bucket_worst_rows.append(case_metric_df.head(10))

        if bucket_worst_rows:
            bucket_worst_df = pd.concat(bucket_worst_rows, ignore_index=True)
            bucket_worst_df = bucket_worst_df[[
                "relation_bucket", "domain", "case", "matched_file_count",
                "entity_f1", "relation_f1", "end2end_f1", "llm_score",
            ]]
            print("\n=== 每个 bucket 最差 case（各取10条）===")
            display(bucket_worst_df.round(4))

    # 4) 全部匹配文件明细表
    detail_rows = []
    for row in model_records:
        detail_rows.append({
            'domain': row.get('domain'),
            'case': row.get('case'),
            'model': row.get('model'),
            'entity_f1': row.get('entity_f1'),
            'relation_f1': row.get('relation_f1'),
            'end2end_f1': row.get('cell_f1'),
            'llm_score': row.get('llm_score'),
            'file': row.get('file'),
        })

    detail_df = pd.DataFrame(detail_rows)
    display(detail_df.round(4))

In [ ]:
# 对应你给的 evaluate.json(约90-102行) 的核心字段
sample_file = Path('dataset/case/base_latest_output/education/case1/education_case1_claude_opus_4_6_only/education_case1_claude_opus_4_6_only_evaluate.json')

with sample_file.open('r', encoding='utf-8') as f:
    sample_data = json.load(f)

sample_metrics = {
    'entity_precision': sample_data.get('entity_precision'),
    'entity_recall': sample_data.get('entity_recall'),
    'entity_f1': sample_data.get('entity_f1'),
    'relation_precision': sample_data.get('relation_precision'),
    'relation_recall': sample_data.get('relation_recall'),
    'relation_f1': sample_data.get('relation_f1'),
    'llm_score': sample_data.get('llm_score'),
}

print('education_case1_claude_opus_4_6_only')
for k, v in sample_metrics.items():
    print(f'{k}: {v}')